# Excel preprocessing playground

Scratch space for trying out how a workbook gets turned into clean per-sheet tables.
The production logic lives in `backend/preprocessing/tables.py`; this notebook calls it so you can poke at each step.
Workbooks under `data/` are gitignored. Write any output files to `development/runs/` (also gitignored).

In [ ]:
import sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "backend").is_dir():  # walk up to the repo root wherever the kernel started
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from backend.preprocessing.tables import normalize_sheet, preprocess_file

pd.set_option("display.max_columns", 50)
WORKBOOK = ROOT / "data" / "document_library" / "documents" / "raw" / "Location of 386 Pokémon in Gen 3.xlsx"  # swap in any .xlsx you want to test
WORKBOOK

## 1. Raw sheets: what pandas sees before any cleanup

In [ ]:
raw_sheets = pd.read_excel(WORKBOOK, sheet_name=None, header=None, dtype=object, keep_default_na=False)
{name: frame.shape for name, frame in raw_sheets.items()}

In [ ]:
sheet = next(iter(raw_sheets))  # pick a sheet by name to inspect
raw_sheets[sheet].head(15)

## 2. One sheet through the normalizer

The header row is detected heuristically. Pass an override like `{"header_row": 3}` or `{"sheet_type": "metadata"}` to see what changes.

In [ ]:
frame, meta = normalize_sheet(sheet, raw_sheets[sheet], override=None)
{k: meta[k] for k in ("sheet_type", "header_row", "row_count", "column_count", "warnings")}

In [ ]:
frame.head(15)

In [ ]:
pd.DataFrame(meta["columns"])

## 3. The whole workbook, as the pipeline produces it

In [ ]:
files, metadata = preprocess_file(WORKBOOK)
pd.DataFrame({name: {k: s[k] for k in ("sheet_type", "header_row", "row_count", "csv_file")}
              for name, s in metadata["sheets"].items()}).T

## 4. The whole workbook as one string

Roughly what a chat tool like Copilot puts in the prompt when you attach the file: every sheet, every non-empty row, as CSV text under a sheet heading.

In [ ]:
def workbook_to_string(sheets):
    parts = []
    for name, frame in sheets.items():
        frame = frame.loc[~(frame == "").all(axis=1)]  # drop fully blank rows
        parts.append(f"## Sheet: {name}\n{frame.to_csv(index=False, header=False).strip()}")
    return "\n\n".join(parts)

workbook_text = workbook_to_string(raw_sheets)
print(f"{len(workbook_text):,} characters")
print(workbook_text[:2000])

### Ask Claude Opus 4.5 with the whole workbook in context

One Bedrock call, no tools: the string above, then the question. Uses `AWS_PROFILE` from `.env` (run `aws sso login` first if it has expired). A call on a large workbook costs real tokens.

In [ ]:
import time

from dotenv import load_dotenv

from backend.agents.llm import LLM
from backend.config.settings import MODELS

load_dotenv(ROOT / ".env")

QUESTION = "Where can I find Ralts?"  # change me

llm = LLM(model_id=MODELS["claude-opus-4.5"], params={"maxTokens": 2048, "temperature": 0.0})
prompt = f"{workbook_text}\n\nBased on the above, {QUESTION}"
start = time.perf_counter()
response = llm.converse("Answer questions using only the spreadsheet content provided.",
                        [{"role": "user", "content": [{"text": prompt}]}])
elapsed = time.perf_counter() - start

usage = response["usage"]
print(f"Input tokens:  {usage['inputTokens']:,}")
print(f"Output tokens: {usage['outputTokens']:,}")
print(f"Time:          {elapsed:.1f}s   (stop reason: {response['stopReason']})\n")
print("\n".join(b["text"] for b in response["output"]["message"]["content"] if "text" in b))

## 5. Your experiments